# Historical water-level data: exploratory analysis

Load the deterministic Phase 3 telemetry and its separate event labels. All stored timestamps are UTC; daily patterns are grouped in `Africa/Casablanca` local time.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

measurements = pd.read_parquet(Path("data/raw/historical_90d.parquet"))
events = pd.read_parquet(Path("data/raw/events_90d.parquet"))
measurements["time"] = pd.to_datetime(measurements["ts"], utc=True)
measurements["local_time"] = measurements["time"].dt.tz_convert("Africa/Casablanca")
measurements["local_hour"] = measurements["local_time"].dt.hour
measurements.shape, events.shape, measurements["time"].min(), measurements["time"].max()

## Daily profile

The hourly level profile reflects the morning and evening consumption peaks. It is not a direct consumption measurement: it includes pump operation, sensor noise, and injected events.

In [ ]:
hourly_level = measurements.groupby("local_hour")["level_pct"].mean()
ax = hourly_level.plot(marker="o", figsize=(10, 4), title="Mean tank level by local hour")
ax.set(xlabel="Hour (Africa/Casablanca)", ylabel="Level (%)", xticks=range(24))
ax.grid(alpha=0.25)
plt.tight_layout()

## Level trace and labelled events

The labels are kept in a separate ground-truth file. Shading the event intervals makes the sensor faults and supply outages visible without adding label columns to the telemetry schema.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(measurements["time"], measurements["level_pct"], linewidth=0.55)
for event_type, group in events.groupby("event_type"):
    for event in group.itertuples(index=False):
        ax.axvspan(
            pd.Timestamp(event.start_time),
            pd.Timestamp(event.end_time),
            alpha=0.12,
            label=event_type,
        )
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), loc="upper right", fontsize="small", ncols=2)
ax.set(title="90-day water level with event intervals", xlabel="UTC", ylabel="Level (%)")
ax.grid(alpha=0.2)
fig.autofmt_xdate()
plt.tight_layout()

## Refill timing

Pump rising edges are used as refill starts. The intervals include simulated demand variation and outages; inspect the shaded event plot when interpreting long gaps.

In [ ]:
pump = measurements["pump_on"].astype(bool)
refill_starts = measurements.loc[pump & ~pump.shift(1, fill_value=False), "time"]
intervals_minutes = refill_starts.diff().dt.total_seconds().dropna() / 60
ax = intervals_minutes.hist(bins=20, figsize=(8, 4))
ax.set(title="Time between pump starts", xlabel="Minutes", ylabel="Refill count")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()